# Qwen2.5-VL — Visual Genome Scene Summarization

Generate comprehensive text summaries of Visual Genome images using **Qwen2.5-VL-7B-Instruct**.
These summaries are designed to be rich enough for downstream question-answering without re-accessing the image.

## Pipeline
1. Load Qwen2.5-VL-7B-Instruct from local checkpoint
2. Load Visual Genome `region_descriptions` split (images + metadata)
3. Prompt the model to produce a detailed scene summary per image
4. Display image vs. generated summary side-by-side

---
## 1. Setup & Installation

In [ ]:
!pip install -q datasets matplotlib Pillow transformers accelerate qwen-vl-utils torch torchvision

In [ ]:
import os
import random
import textwrap
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Reproducibility
random.seed(42)

# Paths
MODEL_PATH = "/home/shaswata/Desktop/models/Qwen/Qwen2.5-VL-7B-Instruct"
DATA_DIR = Path("../data/visual_genome")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = str(DATA_DIR / "hf_cache")

NUM_SAMPLES = 6  # Number of images to process

print("Setup complete.")

---
## 2. Load Qwen2.5-VL-7B-Instruct

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# Load model in bfloat16 for efficiency
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print(f"\nModel loaded: {MODEL_PATH}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

---
## 3. Load Visual Genome Dataset Samples

In [ ]:
# Load region_descriptions subset (contains images + localized captions)
ds = load_dataset(
    "visual_genome",
    "region_descriptions_v1.2.0",
    split="train",
    cache_dir=CACHE_DIR,
)

print(f"Dataset size: {len(ds):,} images")
print(f"Columns: {ds.column_names}")

# Sample random indices
sample_indices = random.sample(range(len(ds)), NUM_SAMPLES)
samples = [ds[i] for i in sample_indices]

print(f"\nSampled {NUM_SAMPLES} images: {[s['image_id'] for s in samples]}")

---
## 4. Generate Comprehensive Scene Summaries

The prompt instructs the model to produce a detailed, structured summary covering:
- **Scene overview** — what is happening, where, and when
- **Objects & attributes** — every visible object with its properties
- **Spatial relationships** — how objects relate to each other
- **Actions & interactions** — what people/animals are doing
- **Background & environment** — weather, lighting, setting details

This level of detail ensures the summary alone is sufficient for answering diverse visual questions.

In [ ]:
SUMMARY_PROMPT = """Describe this image in comprehensive detail so that someone could answer any question about it without seeing the image. Include:

1. **Scene Overview**: What is the main scene? Describe the setting, time of day, and overall atmosphere.
2. **Objects & Attributes**: List every visible object with its color, size, material, condition, and any text/labels.
3. **Spatial Layout**: Describe where objects are relative to each other (left/right, foreground/background, on top of, next to).
4. **People & Actions**: Describe any people — their appearance, clothing, posture, facial expressions, and what they are doing.
5. **Interactions & Relationships**: How are objects and people interacting? What is being used, held, worn, or operated?
6. **Background & Environment**: Describe the environment — indoor/outdoor, weather, lighting, architectural details, vegetation, sky.

Be factual and specific. Do not speculate beyond what is visible."""


def generate_summary(image: Image.Image) -> str:
    """Generate a comprehensive scene summary for a single image."""
    # Ensure RGB
    if image.mode != "RGB":
        image = image.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": SUMMARY_PROMPT},
            ],
        }
    ]

    # Prepare inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    # Decode only generated tokens (skip input tokens)
    generated_ids = output_ids[0, inputs.input_ids.shape[1]:]
    summary = processor.decode(generated_ids, skip_special_tokens=True).strip()
    return summary


print("Summary generation function ready.")

In [ ]:
# Generate summaries for all sampled images
results = []

for i, sample in enumerate(samples):
    image = sample["image"]
    image_id = sample["image_id"]
    num_regions = len(sample["regions"])

    print(f"[{i+1}/{NUM_SAMPLES}] Generating summary for image {image_id} "
          f"({image.size[0]}x{image.size[1]}, {num_regions} regions)...")

    summary = generate_summary(image)
    results.append({
        "image_id": image_id,
        "image": image,
        "summary": summary,
        "num_regions": num_regions,
        "size": f"{image.size[0]}x{image.size[1]}",
    })

    # Print a short preview
    preview = summary[:150] + "..." if len(summary) > 150 else summary
    print(f"   → {preview}\n")

print(f"Done! Generated {len(results)} summaries.")

---
## 5. Side-by-Side Visualization: Image vs. Generated Summary

In [ ]:
def display_results(results, cols=2):
    """Display image vs. generated summary side-by-side in a grid."""
    n = len(results)
    rows = (n + cols - 1) // cols

    fig = plt.figure(figsize=(24, 8 * rows))
    # Each result takes 2 columns: image + text
    gs = gridspec.GridSpec(rows, cols * 2, figure=fig, hspace=0.4, wspace=0.05)

    for idx, res in enumerate(results):
        row = idx // cols
        col = idx % cols

        # Image panel
        ax_img = fig.add_subplot(gs[row, col * 2])
        ax_img.imshow(res["image"])
        ax_img.set_title(
            f"Image {res['image_id']}  ({res['size']})\n"
            f"{res['num_regions']} region descriptions",
            fontsize=11, fontweight="bold", pad=8,
        )
        ax_img.axis("off")

        # Text panel
        ax_txt = fig.add_subplot(gs[row, col * 2 + 1])
        ax_txt.axis("off")

        # Wrap text to fit the panel
        wrapped = textwrap.fill(res["summary"], width=65)
        ax_txt.text(
            0.02, 0.98, wrapped,
            transform=ax_txt.transAxes,
            fontsize=9,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f4ff", edgecolor="#b0b8d0", alpha=0.9),
            wrap=True,
        )
        ax_txt.set_title("Generated Summary", fontsize=11, fontweight="bold", pad=8)

    plt.suptitle(
        "Qwen2.5-VL-7B-Instruct — Visual Genome Scene Summaries",
        fontsize=16, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.show()


display_results(results, cols=2)

---
## 6. Detailed Single-Image View

Zoom into individual results with a larger image and full summary text.

In [ ]:
for res in results:
    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(18, 7),
                                          gridspec_kw={"width_ratios": [1, 1.3]})

    # Image
    ax_img.imshow(res["image"])
    ax_img.set_title(f"Image {res['image_id']}  ({res['size']})", fontsize=13, fontweight="bold")
    ax_img.axis("off")

    # Summary text
    ax_txt.axis("off")
    wrapped = textwrap.fill(res["summary"], width=80)
    ax_txt.text(
        0.02, 0.98, wrapped,
        transform=ax_txt.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="sans-serif",
        linespacing=1.4,
        bbox=dict(boxstyle="round,pad=0.6", facecolor="#f8f9fa", edgecolor="#dee2e6"),
    )
    ax_txt.set_title("Generated Summary (for QA)", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
    print(f"{'─' * 90}\n")

---
## 7. Summary Statistics

In [ ]:
# Quick statistics on generated summaries
lengths = [len(r["summary"]) for r in results]
word_counts = [len(r["summary"].split()) for r in results]

print("Generated Summary Statistics")
print("=" * 40)
print(f"  Samples processed : {len(results)}")
print(f"  Avg char length   : {sum(lengths)/len(lengths):.0f}")
print(f"  Avg word count    : {sum(word_counts)/len(word_counts):.0f}")
print(f"  Min / Max words   : {min(word_counts)} / {max(word_counts)}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(
    [f"IMG {r['image_id']}" for r in results],
    word_counts,
    color="#5b8def",
    edgecolor="#3a5fad",
)
ax.set_xlabel("Word Count")
ax.set_title("Summary Length per Image")
plt.tight_layout()
plt.show()